In [1]:
import os
import re
import string


import pandas as pd
import numpy as np

# Preprocesados
import spacy
import en_core_web_sm

# Vectorizaciones
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import gensim.downloader as api
from gensim.models import KeyedVectors
from transformers import BertTokenizer, BertModel
import torch


In [2]:
# Carga de datos de train
nlp = en_core_web_sm.load()

df_train = None
preprocesado = False

if not os.path.exists("data/train_preprocesado.jsonl"):
    df_train = pd.read_json("data/train.jsonl",lines=True)
else:
    print("Cargando datos preprocesados...")
    df_train = pd.read_json("data/train_preprocesado.jsonl",lines=True)
    preprocesado = True

Cargando datos preprocesados...


## Preprocesado

In [3]:
# Métodos de preprocesado de texto

translator = str.maketrans('', '', string.punctuation)


def limpieza_comun(series_de_texto: pd.Series) -> pd.Series:
    """
    Aplica limpieza común a toda una Serie de Pandas usando métodos vectorizados .str
    """
    # El dataset contiene '\' que deben ser eliminadas manualmente
    texto_limpio = series_de_texto.str.replace(r'\\', ' ', regex=True)
    
    # Eliminar espacios múltiples y aplicar strip()
    texto_limpio = texto_limpio.str.replace(r'\s+', ' ', regex=True).str.strip()
    
    return texto_limpio

def preprocessing_minimal(series_de_texto: pd.Series) -> pd.Series:
    """
    Mantiene puntuación, mayúsculas y estructura.
    Útil para sistemas más complejos como LLMs y Transformers.
    """
    return limpieza_comun(series_de_texto)

def preprocessing_basico(series_de_texto: pd.Series) -> pd.Series:
    """
    Limpia el texto convirtiéndolo a minúsculas, eliminando signos de puntuación
    y lematizando todas las palabras. Conserva las stopwords.
    Útil para TF-IDF o Word2Vec.
    """ 
    textos_limpios = []
    
    for doc in nlp.pipe(limpieza_comun(series_de_texto), 
                        batch_size=1000, 
                        disable=["ner", "parser"]):
        
        # Lematizamos, pasamos a minúscula y quitamos puntuación.
        tokens = [word.lemma_.lower() for word in doc if not word.is_punct]
        
        textos_limpios.append(" ".join(tokens))
        
    # Devolvemos un pd.Series con los índices originales intactos
    return pd.Series(textos_limpios, index=series_de_texto.index)

def preprocessing_avanzado(series_de_texto: pd.Series) -> list:
    """
     Limpia el texto convirtiéndolo todo a minúsculas y eliminando signos de puntuación.
    También elimina las palabras que sean stopWords.
    Útil para modelos estadísticos básados en frecuencias. 
    """
    textos_limpios = []
    
    for doc in nlp.pipe(limpieza_comun(series_de_texto), 
                        batch_size=1000, 
                        disable=["ner", "parser"]):
        
        tokens = [word.lemma_.lower() for word in doc 
                  if not word.is_stop 
                  and not word.is_punct 
                  and word.is_alpha]
        
        textos_limpios.append(" ".join(tokens))
        
    return textos_limpios


In [4]:
if not preprocesado:
    print("Minimo iniziando...",end=" ")
    df_train['desc_minimal'] = preprocessing_minimal(df_train.description)
    print("Minimo Hecho!!")
    print("Básico iniziando...",end=" ")
    df_train['desc_basico'] = preprocessing_basico(df_train.description)
    print("Básico Hecho!!")
    print("Avanzado iniziando...",end=" ")
    df_train['desc_avanzado'] = preprocessing_avanzado(df_train.description)
    print("Avanzado Hecho!!")

    # Ya no se va a usar la descripción original. La descripción orignal ahora será la minimal.
    df_train = df_train.drop(["description"], axis=1)
    df_train.to_json(
        'data/train_preprocesado.jsonl', 
        orient='records', 
        lines=True, 
        force_ascii=False
    )
else:
    print("Datos ya limpios")
    print("Saltando preprocesado...")

Datos ya limpios
Saltando preprocesado...


## Vectorizaciones

In [5]:
PATH_MODELS = os.path.abspath(os.getcwd()) + "/models"
os.makedirs(PATH_MODELS, exist_ok=True) # Asegurar que existe la carpeta
PATH_MODELS

'/home/jcarreno/UNI/PLN_PF/models'

In [6]:
# Descargar el modelo en caso de ser necesario

ruta_modelo_gz = os.path.abspath(os.getcwd()) + "/models/word2vec-google-news-300/word2vec-google-news-300.gz"
w2v = None
if os.path.exists(ruta_modelo_gz):
    print("Modelo ya descargado. Cargando...")
    
    # Cargamos el formato original en crudo, saltándonos el api.load()
    w2v = KeyedVectors.load_word2vec_format(ruta_modelo_gz, binary=True)
    print("Modelo cargado con éxito.")
else:
    os.environ["GENSIM_DATA_DIR"] = PATH_MODELS
    api.BASE_DIR = PATH_MODELS 
    model_name = "word2vec-google-news-300"
    print(f"Descargando {model_name}...")
    w2v = api.load(model_name)
    print(f"{model_name} descargado.")

Modelo ya descargado. Cargando...
Modelo cargado con éxito.


In [7]:
# Vectorización Dispersa vs Densa

# Dispersas

def BoW(textos: pd.Series) -> pd.DataFrame:
    """
    Convierte todos los textos en un Bag Of Words
    limitado a las palabras más frecuentes de los textos.
    """
    # max_features limita a las N palabras más frecuentes
    vectorizador = CountVectorizer(max_features=5000, min_df=2, stop_words='english')
    
    X = vectorizador.fit_transform(textos)
    
    return pd.DataFrame(X.toarray(), columns=vectorizador.get_feature_names_out())

def TF_IDF(textos: pd.Series) -> pd.DataFrame:
    """
    Convierte todos los textos en una matriz con TF-IDF.
    """
    vectorizador = TfidfVectorizer(max_features=5000, min_df=2, stop_words='english')
    
    X = vectorizador.fit_transform(textos)
    
    return pd.DataFrame(X.toarray(), columns=vectorizador.get_feature_names_out())


# Densas

def Word2Vec_Embeddings(textos: pd.Series, modelo_w2v) -> pd.DataFrame:
    """
    Calcula el vector promedio de cada documento usando un modelo Word2Vec preentrenado.
    Recibe la serie de textos y el modelo ya cargado en memoria.
    Recomendación: Pasar 'desc_basico'.
    """
    descripciones_tokenizadas = textos.str.split().tolist()

    dimensiones = modelo_w2v.vector_size 
    
    def obtener_vector_documento(tokens):
        # Filtrar palabras existentes en el vocabulario preentrenado
        palabras_conocidas = [word for word in tokens if word in modelo_w2v]
        
        if not palabras_conocidas:
            # Si ninguna palabra del texto está en el modelo
            return np.zeros(dimensiones)
        
        # Media de los embeddings de las palabras
        return np.mean(modelo_w2v[palabras_conocidas], axis=0)

    # Aplicar a todas las descripciones
    vectores_documentos = [obtener_vector_documento(tokens) for tokens in descripciones_tokenizadas]
    
    nombres_columnas = [f"w2v_google_{i}" for i in range(dimensiones)]
    return pd.DataFrame(vectores_documentos, index=textos.index, columns=nombres_columnas)


def BERT_Embeddings_HF(series_de_texto: pd.Series) -> pd.DataFrame:
    """
    Utiliza BertModel nativo para extraer embeddings contextuales.
    Toma el token especial [CLS] como representación semántica del texto completo.
    """
    print("Cargando modelo BERT y tokenizador...")
    # Cargamos la versión uncased (todo minúsculas por defecto en BERT)
    nombre_modelo = 'bert-base-uncased'
    tokenizer = BertTokenizer.from_pretrained(nombre_modelo, cache_dir=PATH_MODELS)
    modelo = BertModel.from_pretrained(nombre_modelo, cache_dir=PATH_MODELS)
    
    # Cargar modelo a GPU si hay
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    modelo.to(device)
    modelo.eval()
    
    todos_los_vectores = []
    batch_size = 32
    
    print(f"Generando embeddings en {device.type.upper()}...")
    
    # Procesamiento en batches
    for i in range(0, len(series_de_texto), batch_size):
        batch_textos = series_de_texto.iloc[i : i + batch_size].tolist()
        
        # Tokenización
        # max_length=128 recorta textos largos
        inputs = tokenizer(batch_textos, 
                           padding=True, 
                           truncation=True, 
                           max_length=128, 
                           return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = modelo(**inputs)

            cls_embeddings = outputs.last_hidden_state[:, 0, :]
            todos_los_vectores.append(cls_embeddings.cpu().numpy())
            
    # Juntar todos los batches
    matriz_final = np.vstack(todos_los_vectores)
    
    nombres_columnas = [f"bert_hf_{i}" for i in range(matriz_final.shape[1])]
    return pd.DataFrame(matriz_final, index=series_de_texto.index, columns=nombres_columnas)


In [ ]:
df_bow = BoW(df_train.desc_avanzado)
df_tfidf = TF_IDF(df_train.desc_basico)

In [ ]:
print("Palabras más comunes en BoW")
print(df_bow.sum(axis=0).sort_values(ascending=False)[:5])
print()
print("Palabras más representativas en TF-IDF")
print(df_tfidf.sum(axis=0).sort_values(ascending=False)[:5])

Palabras más comunes en BoW
say        24064
new        17533
year       13236
reuters    13006
company     9585
dtype: int64

Palabras más representativas en TF-IDF
39         3272.939778
say        2645.232631
new        2054.659864
year       1750.427724
reuters    1731.556139
dtype: float64


In [ ]:
muestra = df_train.head(5)

print("Word2Vec")

df_w2v_muestra = Word2Vec_Embeddings(muestra['desc_basico'], w2v)

print(f"Forma de la matriz: {df_w2v_muestra.shape}") 
display(df_w2v_muestra.head(2)) 


print("BERT")
df_bert_muestra = BERT_Embeddings_HF(muestra['desc_minimal'])

print(f"Forma de la matriz: {df_bert_muestra.shape}")
display(df_bert_muestra.head(2))